In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
openalex_harvester.py
=====================
A configurable OpenAlex /works harvester for science-of-science studies and
biomedical literature mining.

Key features
------------
- API-key auth (REQUIRED by OpenAlex since 2026-02-13; the old mailto/polite
  pool is gone). Set OPENALEX_API_KEY env var or pass api_key=...
- Complex, composable filters (dict or raw string) + full-text `search`.
- Cursor pagination -> handles arbitrarily large result sets (>10k).
- `select=` to fetch only needed fields (saves credits + payload).
- Nested JSON -> flat `oa_*` schema ready for pandas / DuckDB / PyArrow.
- Abstract reconstruction from OpenAlex's inverted index.
- Retry/backoff on 429/5xx; graceful stop + checkpoint on credit exhaustion.
- Checkpointed parquet output: resumable large harvests.

Quick start
-----------
    from openalex_harvester import OpenAlexHarvester

    h = OpenAlexHarvester(api_key="YOUR_KEY")           # or env OPENALEX_API_KEY
    df = h.harvest(
        filters={"publication_year": "2015-2025", "type": "article"},
        search="tick-borne disease climate change",
        outdir="./oa_ticks",
        max_records=5000,
    )
    print(df.shape, df.columns.tolist())

Requires: requests, pandas, pyarrow
    pip install requests pandas pyarrow
"""

from __future__ import annotations
import os
import re
import json
import time
import glob
import math
from pathlib import Path
from typing import Iterable
from urllib.parse import quote_plus

import requests
import pandas as pd

OPENALEX_WORKS = "https://api.openalex.org/works"

# OPTIONAL: pass this to `select=` ONLY if you want smaller payloads. Since the
# 2025 "Walden" API rewrite, several fields (grants, mesh, fwci,
# sustainable_development_goals, citation_normalized_percentile, ...) are NO
# LONGER valid in `select` even though they're still in the full record body.
# OpenAlex bills per REQUEST, not per field, so omitting select costs no extra
# credits — it only enlarges payloads. Default is None (fetch full records),
# which is the robust choice. Only these are confirmed select-safe:
OPTIONAL_SELECT = [
    "id", "doi", "ids", "title", "display_name",
    "publication_year", "publication_date", "language", "type", "type_crossref",
    "authorships", "primary_location", "open_access", "indexed_in",
    "cited_by_count", "biblio", "primary_topic", "topics", "concepts",
    "keywords", "referenced_works", "referenced_works_count", "related_works",
    "abstract_inverted_index",
]


def short_id(url: str | None) -> str:
    """https://openalex.org/W123 -> W123 ; also trims doi/orcid/ror prefixes."""
    if not url:
        return ""
    return re.sub(r"^https?://(openalex\.org|doi\.org|orcid\.org|ror\.org)/", "", url)


def clean_pmid(url: str | None) -> str:
    """https://pubmed.ncbi.nlm.nih.gov/34567890 -> 34567890"""
    if not url:
        return ""
    m = re.search(r"(\d+)/?$", url)
    return m.group(1) if m else url


def clean_pmcid(url: str | None) -> str:
    """.../pmc/articles/PMC7654321 -> PMC7654321"""
    if not url:
        return ""
    m = re.search(r"(PMC\d+)", url)
    return m.group(1) if m else url


def reconstruct_abstract(inv: dict | None) -> str | None:
    """Turn an abstract_inverted_index {word: [pos,...]} back into text."""
    if not inv:
        return None
    positions: list[tuple[int, str]] = []
    for word, idxs in inv.items():
        for i in idxs:
            positions.append((i, word))
    if not positions:
        return None
    positions.sort(key=lambda x: x[0])
    return " ".join(w for _, w in positions)


# ---------------------------------------------------------------------------
# Keyword gazetteer (defaults — swap in your own groups). These feed a boolean
# OR search over title + abstract + fulltext. NOTE: "tick" and friends are
# homonym magnets (tick box, stock tick, Lyme tick); that noise is expected and
# is what your downstream BERTopic homonym filtering removes.
# ---------------------------------------------------------------------------
TICK_GENERIC = [
    "tick", "ticks", "tick-borne", "tick-bite", "tick bite",
    "tick infestation", "questing tick", "Ixodida",
]
TICK_FAMILY_TERMS = [
    "Ixodidae", "Argasidae", "Nuttalliellidae",
    "hard tick", "hard ticks", "soft tick", "soft ticks",
]
TICK_GENERA = [
    "Ixodes", "Amblyomma", "Rhipicephalus", "Boophilus", "Dermacentor",
    "Haemaphysalis", "Hyalomma", "Ornithodoros", "Argas", "Otobius",
    "Bothriocroton", "Nosomma", "Margaropus", "Rhipicentor", "Nuttalliella",
    "Carios", "Anomalohimalaya", "Cosmiomma", "Aponomma", "Antricola",
    "Nothoaspis",
]


def build_boolean_or(terms: Iterable[str]) -> str:
    """
    Compose an OpenAlex boolean OR search string. Multi-word / hyphenated terms
    are wrapped in double quotes so they match as phrases (unquoted multi-word
    terms would be AND-ed). e.g. ['tick', 'hard tick'] -> '(tick OR "hard tick")'
    """
    parts = []
    for t in terms:
        t = (t or "").strip()
        if not t:
            continue
        parts.append(f'"{t}"' if (" " in t or "-" in t) else t)
    return "(" + " OR ".join(parts) + ")"


def _dedupe(seq: Iterable[str]) -> list[str]:
    """Order-preserving dedupe, dropping falsy values."""
    seen, out = set(), []
    for x in seq:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out


class OpenAlexHarvester:
    def __init__(
        self,
        api_key: str | None = None,
        select: list[str] | None = None,   # None = fetch full records (robust)
        per_page: int = 200,
        sep: str = " | ",
        include_xpac: bool = False,
        max_retries: int = 6,
        timeout: int = 60,
        pause: float = 0.15,  # polite gap between calls (<=10 req/s cap)
    ):
        self.api_key = api_key or os.environ.get("OPENALEX_API_KEY")
        if not self.api_key:
            print("WARNING: no API key set. OpenAlex requires one since "
                  "2026-02-13; without it you get ~100 test credits then 409s. "
                  "Get a free key at https://openalex.org/settings/api")
        self.select = select   # None => no select param => full records
        self.per_page = min(per_page, 200)
        self.sep = sep
        self.include_xpac = include_xpac
        self.max_retries = max_retries
        self.timeout = timeout
        self.pause = pause
        self.session = requests.Session()

    # ---- filter building -------------------------------------------------
    @staticmethod
    def build_filter(filters: dict | str | None) -> str | None:
        """
        dict -> OpenAlex filter string.
          - comma between keys  = AND
          - list value          = OR (joined with '|')
        e.g. {"publication_year": "2015-2025",
              "type": "article",
              "topics.id": ["T10091", "T12345"]}
          -> "publication_year:2015-2025,type:article,topics.id:T10091|T12345"
        A raw string is passed through unchanged.
        """
        if filters is None or isinstance(filters, str):
            return filters
        parts = []
        for k, v in filters.items():
            if isinstance(v, (list, tuple, set)):
                v = "|".join(str(x) for x in v)
            parts.append(f"{k}:{v}")
        return ",".join(parts)

    # ---- HTTP with retry -------------------------------------------------
    def _get(self, params: dict) -> dict:
        params = dict(params)
        if self.api_key:
            params["api_key"] = self.api_key
        if self.include_xpac:
            params["include_xpac"] = "true"
        for attempt in range(self.max_retries):
            try:
                r = self.session.get(OPENALEX_WORKS, params=params, timeout=self.timeout)
            except requests.RequestException as e:
                wait = 2 ** attempt
                print(f"  network error ({e}); retry in {wait}s")
                time.sleep(wait)
                continue
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:  # rate limited
                wait = 2 ** attempt
                print(f"  429 rate-limited; backing off {wait}s")
                time.sleep(wait)
                continue
            if r.status_code in (401, 403, 409):  # auth / credit exhaustion
                raise RuntimeError(
                    f"HTTP {r.status_code} from OpenAlex — likely missing/invalid "
                    f"API key or daily credits exhausted. Body: {r.text[:300]}"
                )
            if 500 <= r.status_code < 600:
                wait = 2 ** attempt
                print(f"  server {r.status_code}; retry in {wait}s")
                time.sleep(wait)
                continue
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
        raise RuntimeError("max retries exceeded")

    # ---- flattening ------------------------------------------------------
    def flatten(self, w: dict) -> dict:
        ids = w.get("ids") or {}
        authorships = w.get("authorships") or []

        authors, orcids, corr = [], [], []
        raw_affils, inst_names, inst_ids, inst_rors, inst_types, countries = [], [], [], [], [], []
        for a in authorships:
            au = a.get("author") or {}
            name = au.get("display_name") or ""
            authors.append(name)
            orcids.append(short_id(au.get("orcid")))            # aligned with authors
            if a.get("is_corresponding"):
                corr.append(name)
            raw_affils.extend(a.get("raw_affiliation_strings") or [])
            for inst in (a.get("institutions") or []):
                inst_names.append(inst.get("display_name") or "")
                inst_ids.append(short_id(inst.get("id")))
                inst_rors.append(short_id(inst.get("ror")))
                inst_types.append(inst.get("type") or "")
                if inst.get("country_code"):
                    countries.append(inst["country_code"])
            countries.extend(a.get("countries") or [])

        ploc = w.get("primary_location") or {}
        source = (ploc.get("source") or {}) if ploc else {}
        issn = source.get("issn_l") or (source.get("issn") or [None])[0]

        pt = w.get("primary_topic") or {}
        pt_sub = (pt.get("subfield") or {})
        pt_field = (pt.get("field") or {})
        pt_dom = (pt.get("domain") or {})

        grants = w.get("grants") or []
        oa = w.get("open_access") or {}
        biblio = w.get("biblio") or {}
        cnp = w.get("citation_normalized_percentile") or {}
        cnp_val = cnp.get("value") if isinstance(cnp, dict) else cnp

        def join(seq):
            return self.sep.join(_dedupe(seq))

        return {
            # --- identifiers ---
            "oa_id": short_id(w.get("id")),
            "oa_doi": short_id(w.get("doi") or ids.get("doi")),
            "oa_pmid": clean_pmid(ids.get("pmid")),
            "oa_pmcid": clean_pmcid(ids.get("pmcid")),
            # --- core text ---
            "oa_title": w.get("title") or w.get("display_name"),
            "oa_abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
            # --- people ---
            "oa_authors": self.sep.join(authors),          # keep order + dupes for alignment
            "oa_author_orcids": self.sep.join(orcids),      # index-aligned with oa_authors
            "oa_corresponding_authors": join(corr),
            "oa_authors_count": len(authors),
            # --- affiliations / geography ---
            "oa_affiliations_raw": join(raw_affils),
            "oa_institutions": join(inst_names),
            "oa_institution_ids": join(inst_ids),
            "oa_institution_rors": join(inst_rors),
            "oa_institution_types": join(inst_types),
            "oa_countries": join(countries),
            # --- impact ---
            "oa_cited_by_count": w.get("cited_by_count"),
            "oa_fwci": w.get("fwci"),
            "oa_citation_pct": cnp_val,
            "oa_referenced_works": join(short_id(x) for x in (w.get("referenced_works") or [])),
            "oa_referenced_works_count": w.get("referenced_works_count"),
            # --- venue ---
            "oa_journal": source.get("display_name"),
            "oa_issn": issn,
            "oa_type": w.get("type"),
            "oa_language": w.get("language"),
            "oa_volume": biblio.get("volume"),
            "oa_issue": biblio.get("issue"),
            "oa_first_page": biblio.get("first_page"),
            # --- funding ---
            "oa_funders": join(g.get("funder_display_name") for g in grants),
            "oa_funder_ids": join(short_id(g.get("funder")) for g in grants),
            "oa_award_ids": join(g.get("award_id") for g in grants),
            "oa_has_funding": bool(grants),
            # --- dates ---
            "oa_year": w.get("publication_year"),
            "oa_publication_date": w.get("publication_date"),
            # --- classification ---
            "oa_topic": pt.get("display_name"),
            "oa_subfield": pt_sub.get("display_name"),
            "oa_field": pt_field.get("display_name"),
            "oa_domain": pt_dom.get("display_name"),
            "oa_topics_all": join(
                f"{t.get('display_name')} ({round(t.get('score', 0), 3)})"
                for t in (w.get("topics") or [])
            ),
            "oa_concepts": join(c.get("display_name") for c in (w.get("concepts") or [])),
            "oa_keywords": join(k.get("display_name") for k in (w.get("keywords") or [])),
            "oa_mesh": join(m.get("descriptor_name") for m in (w.get("mesh") or [])),
            "oa_sdgs": join(
                s.get("display_name") for s in (w.get("sustainable_development_goals") or [])
            ),
            # --- open access / integrity ---
            "oa_is_oa": oa.get("is_oa"),
            "oa_oa_status": oa.get("oa_status"),
            "oa_is_retracted": w.get("is_retracted"),
            "oa_indexed_in": self.sep.join(w.get("indexed_in") or []),
        }

    # ---- count (cheap size check before a full harvest) -----------------
    def count(self, filters: dict | str | None = None, search: str | None = None) -> int | None:
        """One request that returns meta.count — use this to size a corpus
        (and estimate credits) BEFORE committing to a full harvest."""
        params = {"per-page": 1}
        filt = self.build_filter(filters)
        if filt:
            params["filter"] = filt
        if search:
            params["search"] = search
        data = self._get(params)
        return data.get("meta", {}).get("count")

    # ---- harvest ---------------------------------------------------------
    def harvest(
        self,
        filters: dict | str | None = None,
        search: str | None = None,
        outdir: str = "./openalex_out",
        max_records: int | None = None,
        resume: bool = True,
        flush_every_pages: int = 10,
    ) -> pd.DataFrame:
        """
        Harvest works matching `filters` (+ optional full-text `search`) and
        write checkpointed parquet parts into `outdir`. Returns the combined
        DataFrame.
        """
        out = Path(outdir)
        out.mkdir(parents=True, exist_ok=True)
        state_file = out / "_cursor.txt"
        filt = self.build_filter(filters)

        # credit estimate (list call = ~10 credits)
        if max_records:
            est_pages = math.ceil(max_records / self.per_page)
            print(f"Estimate: up to ~{est_pages} list calls "
                  f"(~{est_pages * 10} credits). Free tier ~$1/day.")

        base = {"per-page": self.per_page}
        if self.select:
            base["select"] = ",".join(self.select)
        if filt:
            base["filter"] = filt
        if search:
            base["search"] = search

        cursor = "*"
        if resume and state_file.exists():
            saved = state_file.read_text().strip()
            if saved and saved != "DONE":
                cursor = saved
                print(f"Resuming from saved cursor.")
            elif saved == "DONE":
                print("State says DONE; re-combining existing parts.")
                return self._combine(out)

        existing = sorted(glob.glob(str(out / "part_*.parquet")))
        part_no = len(existing)
        buffer: list[dict] = []
        total = sum(len(pd.read_parquet(p, columns=["oa_id"])) for p in existing) if existing else 0
        page = 0

        while cursor:
            params = dict(base, cursor=cursor)
            data = self._get(params)
            results = data.get("results", [])
            if not results:
                break
            buffer.extend(self.flatten(w) for w in results)
            total += len(results)
            page += 1
            cursor = data.get("meta", {}).get("next_cursor")

            if page % flush_every_pages == 0 and buffer:
                self._flush(buffer, out, part_no)
                part_no += 1
                buffer = []
                state_file.write_text(cursor or "DONE")
                print(f"  ...{total} records harvested")

            if max_records and total >= max_records:
                break
            time.sleep(self.pause)

        if buffer:
            self._flush(buffer, out, part_no)
        state_file.write_text("DONE")
        print(f"Done. {total} records in {outdir}")
        return self._combine(out)

    @staticmethod
    def _flush(buffer: list[dict], out: Path, part_no: int) -> None:
        df = pd.DataFrame(buffer)
        df.to_parquet(out / f"part_{part_no:05d}.parquet", index=False)

    @staticmethod
    def _combine(out: Path) -> pd.DataFrame:
        parts = sorted(glob.glob(str(out / "part_*.parquet")))
        if not parts:
            return pd.DataFrame()
        df = pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True)
        df = df.drop_duplicates(subset=["oa_id"]).reset_index(drop=True)
        df.to_parquet(out / "combined.parquet", index=False)
        return df

    # ---- keyword-driven harvest -----------------------------------------
    @staticmethod
    def chunk_terms(terms: list[str], budget: int = 3500) -> list[list[str]]:
        """
        Split terms so each resulting boolean OR query stays under `budget`
        URL-ENCODED bytes (OpenAlex caps the whole request URL at ~4094 bytes;
        quotes/parens/spaces inflate once encoded, so we measure the encoded
        length, not the raw string).
        """
        chunks: list[list[str]] = []
        cur: list[str] = []
        for t in terms:
            trial = build_boolean_or(cur + [t])
            if cur and len(quote_plus(trial)) > budget:
                chunks.append(cur)
                cur = [t]
            else:
                cur.append(t)
        if cur:
            chunks.append(cur)
        return chunks

    def harvest_terms(
        self,
        term_groups: dict | list,
        outdir: str = "./openalex_terms",
        extra_filters: dict | None = None,
        from_date: str = "2000-01-01",
        to_date: str | None = None,
        article_only: bool = True,
        max_records: int | None = None,   # NOTE: applied PER chunk (use for tests)
        budget: int = 3500,
    ) -> pd.DataFrame:
        """
        Harvest works matching ANY term across your keyword groups, over
        title+abstract+fulltext, restricted to original articles from `from_date`
        to today. Auto-chunks long term lists to respect the URL limit, then
        combines and dedupes across chunks.

        term_groups: {"TICK_GENERA": [...], "TICK_GENERIC": [...]} OR a list of
                     lists OR a flat list of strings.
        """
        # flatten groups -> unique term list (order-preserving)
        if isinstance(term_groups, dict):
            groups = list(term_groups.values())
        elif term_groups and isinstance(term_groups[0], (list, tuple)):
            groups = list(term_groups)
        else:
            groups = [term_groups]
        terms = _dedupe(t for g in groups for t in g)

        filters = dict(extra_filters or {})
        if from_date:
            filters["from_publication_date"] = from_date
        if to_date:
            filters["to_publication_date"] = to_date
        if article_only:
            filters["type"] = "article"

        chunks = self.chunk_terms(terms, budget=budget)
        print(f"{len(terms)} terms -> {len(chunks)} query chunk(s); "
              f"filters: {self.build_filter(filters)}")

        out = Path(outdir)
        out.mkdir(parents=True, exist_ok=True)
        frames = []
        for i, chunk in enumerate(chunks):
            q = build_boolean_or(chunk)
            print(f"\n[chunk {i + 1}/{len(chunks)}] {len(chunk)} terms, "
                  f"~{len(quote_plus(q))} url-bytes")
            df = self.harvest(
                filters=filters, search=q,
                outdir=str(out / f"chunk_{i:02d}"),
                max_records=max_records,
            )
            frames.append(df)

        combined = (
            pd.concat(frames, ignore_index=True).drop_duplicates("oa_id")
            if frames else pd.DataFrame()
        )
        combined.to_parquet(out / "combined_all.parquet", index=False)
        print(f"\nTOTAL unique works across chunks: {len(combined)}")
        return combined


if __name__ == "__main__":
    # Tick literature, 2000 -> today, original articles only, searched across
    # title + abstract + fulltext using the default tick gazetteer.
    # from openalex_harvester import TICK_GENERIC, TICK_FAMILY_TERMS, TICK_GENERA

    h = OpenAlexHarvester()  # reads OPENALEX_API_KEY from env / Colab secret
    # df = h.harvest_terms(
    #     term_groups={
    #         "TICK_GENERIC": TICK_GENERIC,
    #         "TICK_FAMILY_TERMS": TICK_FAMILY_TERMS,
    #         "TICK_GENERA": TICK_GENERA,
    #     },
    #     outdir="./oa_ticks",
    #     from_date="2000-01-01",   # to today (no to_date)
    #     article_only=True,        # original articles (excludes review/letter/editorial/...)
    #     # max_records=500,        # uncomment to test on a small slice first
    # )
    # print(df.shape)
    # print(df.columns.tolist())
    # # df.to_csv("oa_ticks.csv", index=False)

In [ ]:
from google.colab import userdata
# from openalex_harvester import OpenAlexHarvester

# --- taxon + keyword terms -------------------------------------------------
GENERA = [
    "ixodes","amblyomma","rhipicephalus","boophilus","dermacentor","haemaphysalis",
    "hyalomma","ornithodoros","argas","otobius","carios","nuttalliella",
    "rhipicentor","margaropus","anomalohimalaya","cosmiomma","nosomma",
]
FAMILIES = ["ixodidae","argasidae","ixodida","nuttalliellidae","acari"]
# Quote multiword/hyphenated terms so they aren't split into AND-ed words.
# Stemming means "tick" already matches "ticks", so plurals are redundant.
PHRASES  = ['tick', 'ticks', '"tick-borne"', '"tick borne"']

TERMS = GENERA + FAMILIES + PHRASES
QUERY = " OR ".join(TERMS)
print("query chars:", len(QUERY), "/ ~4000 URL limit")   # ~300, plenty of room

h = OpenAlexHarvester(api_key=userdata.get('OPENALEX_API_KEY'))

FILTERS = {
    "publication_year": "2000-2026",
    # "from_publication_date": "2000-01-01",   # 2000 -> today (auto-includes newest)
    "type": ["article", "review", 'letter', 'preprint'],                        # original research; excludes the 'review' type
    "is_paratext": "false",                  # drop front-matter / non-substantive items
}

# 1) SIZE CHECK FIRST — one call, before spending credits on the full pull
n = h.count(filters=FILTERS, search=QUERY)
print(f"Matching works: {n:,}   (~{-(-n//200)} pages ≈ {-(-n//200)*10:,} credits)")

# 2) Then harvest everything
df = h.harvest(
    filters=FILTERS,
    search=QUERY,
    outdir="/content/drive/MyDrive/COGNOSX/ticks/data_raw",
    max_records=None,        # None = all matches (not just 1000)
)
print(df.shape)

In [ ]:
df

In [ ]:
df.info()